## Uniform Int4@4.0 — 3,000-Step Signal Run

**Goal:** Execute a stable 3,000-step training run on a single T4 GPU to verify BPB stability and artifact compliance (<16MB).

**Settings:** 1024 seq_len · 262k batch tokens · lzma compression · val every 500 steps

**Hardware:** GPU T4 x1 (single GPU, no DDP — proven stable from 700-step proof)

| Cell | Purpose | Est. Time |
|------|---------|----------|
| 1 | Setup: clone repos, install deps, download data | ~3 min |
| 2 | Patch: fix paths + SDPA backend for T4 | ~5 sec |
| 3 | Train: 3,000-step signal run | ~2-3 hrs |

In [ ]:
# Cell 1/3 — Setup: repos + deps + data
import os, subprocess, shutil
from pathlib import Path

os.chdir('/kaggle/working')

# Clean previous clones
for d in ['pg', 'openai-pg']:
    if os.path.exists(d):
        shutil.rmtree(d)

print('=== Cloning repositories ===')
!git clone --depth 1 https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git pg
!git clone --depth 1 https://github.com/openai/parameter-golf.git openai-pg

print('\n=== Installing dependencies ===')
%pip install -q sentencepiece zstandard huggingface_hub

print('\n=== Downloading data shard ===')
os.chdir('/kaggle/working/openai-pg')
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

# Verify essential files exist
tok = Path('/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model')
data = Path('/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024')
script = Path('/kaggle/working/pg/train_gpt.py')
assert tok.exists(), f'Tokenizer not found: {tok}'
assert data.exists(), f'Dataset dir not found: {data}'
assert script.exists(), f'Training script not found: {script}'
print(f'\n✅ SETUP COMPLETE')
print(f'   Tokenizer: {tok}')
print(f'   Dataset:   {data}')
print(f'   Script:    {script}')

In [ ]:
# Cell 2/3 — Patch: Kaggle paths + T4 SDPA backend
from pathlib import Path

train_script = Path('/kaggle/working/pg/train_gpt.py')
content = train_script.read_text()

# --- Patch 1: Redirect data paths to /kaggle/working ---
content = content.replace(
    './data/tokenizers/fineweb_1024_bpe.model',
    '/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model'
)
content = content.replace(
    './data/datasets/fineweb10B_sp1024',
    '/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024'
)

# --- Patch 2: SDPA backend for T4 (compute capability 7.5) ---
# T4 doesn't support flash attention (requires SM>=80).
# The stock code only sets SDPA backends for capability >= 8,
# leaving T4 with all backends enabled (including cuDNN which can fail).
# We add an else-branch for T4 that enables math + mem_efficient only.
old_sdpa = (
    '    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(True)\n'
    '        enable_mem_efficient_sdp(False)\n'
    '        enable_math_sdp(False)\n'
)
new_sdpa = (
    '    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(True)\n'
    '        enable_mem_efficient_sdp(False)\n'
    '        enable_math_sdp(False)\n'
    '    elif torch.cuda.is_available():\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(False)\n'
    '        enable_mem_efficient_sdp(True)\n'
    '        enable_math_sdp(True)\n'
)

if old_sdpa in content:
    content = content.replace(old_sdpa, new_sdpa, 1)
    print('✅ SDPA patch applied (T4 fallback: math + mem_efficient)')
elif 'elif torch.cuda.is_available():' in content and 'enable_math_sdp(True)' in content:
    print('✅ SDPA patch already present')
else:
    print('⚠️  SDPA block not found — check train_gpt.py manually')

train_script.write_text(content)

# Verify patches applied
verify = train_script.read_text()
assert '/kaggle/working/openai-pg/data/tokenizers' in verify, 'Path patch FAILED'
print('✅ Path patches verified')
print('✅ PATCH COMPLETE — ready to train')

In [ ]:
# Cell 3/3 — Train: 3,000-step Signal Run (single T4 GPU)
import os, subprocess, time, gc, torch

# Free any leftover GPU memory from setup cells
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.chdir('/kaggle/working/pg')

env = os.environ.copy()
env.update({
    # === Training Config ===
    'ITERATIONS':         '3000',      # 3k-step Signal Run
    'TRAIN_SEQ_LEN':      '1024',      # Sequence length
    'TRAIN_BATCH_TOKENS': '262144',    # 256 seqs × 1024 tokens
    'WARMUP_STEPS':       '20',        # Standard warmup
    'WARMDOWN_ITERS':     '600',       # ~20% warmdown
    'SEED':               '42',
    # === Validation ===
    'VAL_LOSS_EVERY':     '500',       # Validate every 500 steps
    'VAL_MAX_TOKENS':     '65536',     # Faster validation
    'VAL_BATCH_SIZE':     '524288',
    # === Logging ===
    'TRAIN_LOG_EVERY':    '50',        # Log train loss every 50 steps
    # === Serialization ===
    'COMPRESSOR':         'lzma',      # Best compression ratio
    # === Safety ===
    'MAX_WALLCLOCK_SECONDS': '39600',  # 11-hour safety cap (Kaggle limit = 12h)
    # === Eval ===
    'EVAL_STRIDE':        '64',        # Sliding window stride
    'EVAL_BATCH_SEQS':    '32',
    'EVAL_CACHE':         '0',         # No cache (saves CPU memory)
    # === Environment ===
    'PYTHONUNBUFFERED':   '1',
})

print(f'Starting Uniform Int4@4.0 Signal Run...')
print(f'  ITERATIONS={env["ITERATIONS"]}')
print(f'  TRAIN_SEQ_LEN={env["TRAIN_SEQ_LEN"]}')
print(f'  TRAIN_BATCH_TOKENS={env["TRAIN_BATCH_TOKENS"]}')
print(f'  VAL_LOSS_EVERY={env["VAL_LOSS_EVERY"]}')
print(f'  COMPRESSOR={env["COMPRESSOR"]}')
print(f'  MAX_WALLCLOCK_SECONDS={env["MAX_WALLCLOCK_SECONDS"]}')
print('=' * 60)

t0 = time.time()
# Single GPU — no torch.distributed.run needed
proc = subprocess.run(
    ['python3', '-u', 'train_gpt.py'],
    env=env,
)
elapsed = time.time() - t0

print(f'\n{"=" * 60}')
print(f'Run completed in {elapsed/3600:.2f} hours (exit code: {proc.returncode})')

# Check for artifacts
from pathlib import Path
for f in ['final_model.pt', 'final_model.mixed.ptz', 'final_summary.json', 'final_summary.md']:
    p = Path(f'/kaggle/working/pg/{f}')
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f'  ✅ {f}: {size_mb:.2f} MB')
    else:
        print(f'  ❌ {f}: NOT FOUND')

# Show final summary if available
summary_path = Path('/kaggle/working/pg/final_summary.md')
if summary_path.exists():
    print(f'\n--- Final Summary ---')
    print(summary_path.read_text())